In [2]:
import os
import numpy as np
import pandas as pd

# Modelling 
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

# ==========================================
# 1. LOAD DATA CLEANLY
# ==========================================
file_path = 'data/stud.csv'
if not os.path.exists(file_path):
    raise FileNotFoundError(f"Missing dataset! Please check that '{file_path}' exists.")

df = pd.read_csv(file_path)
print(f"--- Dataset loaded successfully. Total rows: {len(df)} ---")

# ==========================================
# 2. PREPARE X AND Y VARIABLES
# ==========================================
# Target is math_score. Features include demographics and reading/writing scores
X = df.drop(columns=['math_score'])
y = df['math_score']

# ==========================================
# 3. TRANSFORMATION PIPELINE
# ==========================================
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features),        
    ]
)

# Fit and transform the features
X = preprocessor.fit_transform(X)

# ==========================================
# 4. TRAIN TEST SPLIT (80% Train, 20% Test)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]} rows | Test size: {X_test.shape[0]} rows\n")

# ==========================================
# 5. EVALUATION METRICS FUNCTION
# ==========================================
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

# ==========================================
# 6. MODEL TRAINING LOOP (Fast Models Only)
# ==========================================
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)

    # Predict testing data
    y_test_pred = model.predict(X_test)
    
    # Calculate performance metrics
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    # Save results
    model_list.append(name)
    r2_list.append(model_test_r2)
    
    print(f"-> Finished training: {name} (R2 Score: {model_test_r2:.4f})")

# ==========================================
# 7. FINAL COMPARISON REPORT
# ==========================================
print("\n" + "="*45)
print("         FINAL PERFORMANCE REPORT")
print("="*45)
report_df = pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2 Score'])
report_df = report_df.sort_values(by=["R2 Score"], ascending=False).reset_index(drop=True)
print(report_df)

--- Dataset loaded successfully. Total rows: 1000 ---
Train size: 800 rows | Test size: 200 rows

-> Finished training: Linear Regression (R2 Score: 0.8804)
-> Finished training: Lasso (R2 Score: 0.8253)
-> Finished training: Ridge (R2 Score: 0.8806)
-> Finished training: Decision Tree (R2 Score: 0.7206)


C:\Users\KOUSTAV\AppData\Local\Temp\ipykernel_6772\2007033596.py:36: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include="object").columns


-> Finished training: Random Forest Regressor (R2 Score: 0.8517)
-> Finished training: XGBRegressor (R2 Score: 0.8278)
-> Finished training: AdaBoost Regressor (R2 Score: 0.8501)

         FINAL PERFORMANCE REPORT
                Model Name  R2 Score
0                    Ridge  0.880593
1        Linear Regression  0.880433
2  Random Forest Regressor  0.851737
3       AdaBoost Regressor  0.850114
4             XGBRegressor  0.827797
5                    Lasso  0.825320
6            Decision Tree  0.720574
